## Регионы

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re

headers = {
    'accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7',
    'accept-language': 'ru-RU,ru;q=0.9,en-US;q=0.8,en;q=0.7',
    'cache-control': 'max-age=0',
    'priority': 'u=0, i',
    'referer': 'https://www.google.com/',
    'sec-ch-ua': '"Not:A-Brand";v="99", "Google Chrome";v="145", "Chromium";v="145"',
    'sec-ch-ua-mobile': '?0',
    'sec-ch-ua-platform': '"Windows"',
    'sec-fetch-dest': 'document',
    'sec-fetch-mode': 'navigate',
    'sec-fetch-site': 'cross-site',
    'sec-fetch-user': '?1',
    'upgrade-insecure-requests': '1',
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36',
}

response = requests.get(
    'https://ru.wikipedia.org/wiki/%D0%A1%D1%83%D0%B1%D1%8A%D0%B5%D0%BA%D1%82%D1%8B_%D0%A0%D0%BE%D1%81%D1%81%D0%B8%D0%B9%D1%81%D0%BA%D0%BE%D0%B9_%D0%A4%D0%B5%D0%B4%D0%B5%D1%80%D0%B0%D1%86%D0%B8%D0%B8',
    headers=headers,
)

bs = BeautifulSoup(response.content)

data = []
for row in bs.find('table', class_='standard').find('tbody').find_all('tr')[1:]:
    row_data = []
    for col in row.find_all('td'):
        row_data.append(col.text)
    data.append(row_data)

cols = []
row =bs.find('table', class_='standard').find('tbody').find_all('tr')[0]
for col in row.find_all('th'):
    cols.append(col.text)

In [9]:
subjects_df = pd.DataFrame(data, columns=cols)\
    .drop(columns=['Флаг', 'Герб', 'Население01.01.2025[17]', 'История адм.-тер. деления'])
subjects_df

,№,Субъект Российской Федерации,Терри-тория (км²),Адм. центр/столица,Адм.-тер. устройство(согласно ОКАТО),"Код субъекта (ГАИ, МВД)",Код ОКАТО,Муниципальные образования
0,1e-06,Республики,,,,,,
1,1,Республика Адыгея,7792,Майкоп,7 районов и 2 города,01,79,"7 муниципальных районов, 2 городских округа"
2,2,Республика Алтай,92903,Горно-Алтайск,10 районов и 1 город,04,84,"10 муниципальных районов, 1 городской округ"
3,3,Республика Башкортостан,142947,Уфа,54 района и 21 город,02,80,"54 муниципальных района, 9 городских округов"
4,4,Республика Бурятия,351334,Улан-Удэ,21 район и 2 города,03,81,"21 муниципальный район, 2 городских округа"
...,...,...,...,...,...,...,...,...
91,86,Ненецкий АО[11],176810,Нарьян-Мар,1 район и 1 город,83,11,"1 муниципальный район, 1 городской округ"
92,87,Ханты-Мансийский АО — Югра[14],534801,Ханты-Мансийск,9 районов и 14 городов,86,71,"9 муниципальных районов, 13 городских округов"
93,88,Чукотский АО,721481,Анадырь,8 районов и 1 город,87,77,"6 муниципальных районов, 1 городской округ"
94,89,Ямало-Ненецкий АО[14],769250,Салехард,7 районов и 8 городов,89,71,"7 муниципальных районов, 6 городских округов"


In [10]:
subjects_df.loc[1:24, 'subject_type'] = 'Республика'
subjects_df.loc[26:34, 'subject_type'] = 'Край'
subjects_df.loc[36:83, 'subject_type'] = 'Область'
subjects_df.loc[85:87, 'subject_type'] = 'Город федерального назначения'
subjects_df.loc[89:89, 'subject_type'] = 'Автономная область'
subjects_df.loc[91:94, 'subject_type'] = 'Автономный округ'
subjects_df.drop(index=[0, 25, 35, 84, 88, 90, 95], inplace=True)
subjects_df.set_index('№', inplace=True)
subjects_df.rename(columns={
    'Субъект Российской Федерации': 'region',
    'Терри-тория (км²)': 'area',
    'Адм. центр/столица': 'center',
    'Адм.-тер. устройство(согласно ОКАТО)': 'admin_struct',
    'Муниципальные образования': 'municipalities'
}, inplace=True)
subjects_df.loc['79', 'area'] = '26461'
subjects_df['area'] = subjects_df['area'].apply(lambda x: x.split()[0]).astype('int')
subjects_df.replace(r'\[\d+(?:, \d+)*\]', '', regex=True, inplace=True) # Удлаение сссылок
subjects_df.index = subjects_df.index.astype('int')

admin_structs = ['район', 'город', 'пгт', 'ЗАТО', 'административный округ',
                 'муниципальный район', 'муниципальный округ', 'городской округ', 'внутригородское муниципальное образование']
word_to_name = {
    'районов': 'район',
    'района': 'район',
    'городов': 'город',
    'города': 'город',
    'административных округов': 'административный округ',
    'административных округа': 'административный округ',
    'муниципальных районов': 'муниципальный район',
    'муниципальных района': 'муниципальный район',
    'муниципальных округов': 'муниципальный округ',
    'муниципальных округа': 'муниципальный округ',
    'городских округов': 'городской округ',
    'городских округа': 'городской округ',
    'внутригородских муниципальных образований': 'внутригородское муниципальное образование',
}

def extract_struct(s: str):
    # s = s.lower()
    s = re.sub(r'\(.*\)', '', s)
    s = s.replace(' и ', ', ').strip()
    s_arr = s.split(', ')
    admin_structs_num = {}
    for i, e in enumerate(s_arr):
        # print(e)
        num, struct = e.split(' ', maxsplit=1)
        num = int(num)
        struct = word_to_name.get(struct, struct)
        admin_structs_num[struct] = num
    
    # print(admin_structs_num)
    return admin_structs_num

subjects_df[admin_structs] = 0
for i in subjects_df.index:
    extracted_struct = extract_struct(subjects_df.loc[i, 'admin_struct']) | extract_struct(subjects_df.loc[i, 'municipalities'])
    for struct, num in extracted_struct.items():
        subjects_df.loc[i, struct] = num

subjects_df.drop(columns=['admin_struct', 'municipalities'], inplace=True)
subjects_df

,region,area,center,"Код субъекта (ГАИ, МВД)",Код ОКАТО,subject_type,район,город,пгт,ЗАТО,административный округ,муниципальный район,муниципальный округ,городской округ,внутригородское муниципальное образование
№,,,,,,,,,,,,,,,
1,Республика Адыгея,7792,Майкоп,01,79,Республика,7,2,0,0,0,7,0,2,0
2,Республика Алтай,92903,Горно-Алтайск,04,84,Республика,10,1,0,0,0,10,0,1,0
3,Республика Башкортостан,142947,Уфа,02,80,Республика,54,21,0,0,0,54,0,9,0
4,Республика Бурятия,351334,Улан-Удэ,03,81,Республика,21,2,0,0,0,21,0,2,0
5,Республика Дагестан,50270,Махачкала,05,82,Республика,41,10,0,0,0,42,0,10,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,Еврейская АО,36271,Биробиджан,79,99,Автономная область,5,1,0,0,0,5,0,1,0
86,Ненецкий АО,176810,Нарьян-Мар,83,11,Автономный округ,1,1,0,0,0,1,0,1,0
87,Ханты-Мансийский АО — Югра,534801,Ханты-Мансийск,86,71,Автономный округ,9,14,0,0,0,9,0,13,0


## Федеральные округа

In [11]:
response = requests.get(
    'https://ru.wikipedia.org/wiki/%D0%A4%D0%B5%D0%B4%D0%B5%D1%80%D0%B0%D0%BB%D1%8C%D0%BD%D1%8B%D0%B5_%D0%BE%D0%BA%D1%80%D1%83%D0%B3%D0%B0_%D0%A0%D0%BE%D1%81%D1%81%D0%B8%D0%B9%D1%81%D0%BA%D0%BE%D0%B9_%D0%A4%D0%B5%D0%B4%D0%B5%D1%80%D0%B0%D1%86%D0%B8%D0%B8',
    headers=headers,
)

bs = BeautifulSoup(response.content)

In [ ]:
cols = [e.text for e in bs.find('table', class_='wikitable').find('tbody').find_all('tr')[0].find_all('th')]
rows = bs.find('table', class_='wikitable').find('tbody').find_all('tr')[1:]

data = []
for row in rows:
    row_data = []
    for col in row.find_all('td'):
        row_data.append(col.text)
    data.append(row_data)

federals_df = pd.DataFrame(data, columns=cols)
federals_df.rename(
    columns={
        'НазваниеФО': 'region',
        'Площадь(км²)[11][12]\n': 'area',
        'Кол-восуб-овРФ\n': 'количество субъектов',
        'Админ.центр\n': 'center',
    }, inplace=True)

def fix_area(area):
    area = area.replace('&', '').split('.')[0]
    return int(area)

federals_df['area'] = federals_df['area'].apply(fix_area)
federals_df = federals_df[['region', 'area', 'количество субъектов', 'center']]
federals_df.loc[0:7, 'region'] = federals_df.loc[0:7, 'region'] + 'федеральный округ'
federals_df.loc[8, 'region'] = 'Российская Федерация'
federals_df['subject_type'] = 'Федеральный округ'
federals_df['количество субъектов'] = federals_df['количество субъектов'].astype('int')
federals_df['region'] = federals_df['region'].apply(lambda x: x.replace('\n', ' '))
federals_df

,region,area,количество субъектов,center,subject_type
0,Центральный федеральный округ,650205,18,Москва\n,Федеральный округ
1,Северо-Западный федеральный округ,1686972,11,Санкт-Петербург\n,Федеральный округ
2,Южный федеральный округ,447821,8,Ростов-на-Дону\n,Федеральный округ
3,Северо-Кавказский федеральный округ,170439,7,Пятигорск\n,Федеральный округ
4,Приволжский федеральный округ,1036975,14,Нижний Новгород\n,Федеральный округ
5,Уральский федеральный округ,1818497,6,Екатеринбург\n,Федеральный округ
6,Сибирский федеральный округ,4361727,10,Новосибирск\n,Федеральный округ
7,Дальневосточный федеральный округ,6952555,11,Владивосток\n,Федеральный округ
8,Российская Федерация,17125191,83,Москва\n,Федеральный округ


In [13]:
add_federals = pd.DataFrame([
    ('Крымский федеральный округ', 26081, 2, 'Симферополь', 'Федеральный округ')],
    columns=['region', 'area', 'количество субъектов', 'center', 'subject_type']) 
federals_df = pd.concat([federals_df, add_federals])

## Объединение и сохранение

In [14]:
regions_df = pd.concat([subjects_df, federals_df])
regions_df.fillna(-1, inplace=True)
regions_df['region'] = regions_df['region'].apply(lambda x: x.replace('–', '-').replace('—', '-'))
regions_df

,region,area,center,"Код субъекта (ГАИ, МВД)",Код ОКАТО,subject_type,район,город,пгт,ЗАТО,административный округ,муниципальный район,муниципальный округ,городской округ,внутригородское муниципальное образование,количество субъектов
1,Республика Адыгея,7792,Майкоп,01,79,Республика,7.0,2.0,0.0,0.0,0.0,7.0,0.0,2.0,0.0,-1.0
2,Республика Алтай,92903,Горно-Алтайск,04,84,Республика,10.0,1.0,0.0,0.0,0.0,10.0,0.0,1.0,0.0,-1.0
3,Республика Башкортостан,142947,Уфа,02,80,Республика,54.0,21.0,0.0,0.0,0.0,54.0,0.0,9.0,0.0,-1.0
4,Республика Бурятия,351334,Улан-Удэ,03,81,Республика,21.0,2.0,0.0,0.0,0.0,21.0,0.0,2.0,0.0,-1.0
5,Республика Дагестан,50270,Махачкала,05,82,Республика,41.0,10.0,0.0,0.0,0.0,42.0,0.0,10.0,0.0,-1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5,Уральский федеральный округ,1818497,Екатеринбург\n,-1,-1,Федеральный округ,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,6.0
6,Сибирский федеральный округ,4361727,Новосибирск\n,-1,-1,Федеральный округ,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,10.0
7,Дальневосточный федеральный округ,6952555,Владивосток\n,-1,-1,Федеральный округ,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,11.0
8,Российская Федерация,17125191,Москва\n,-1,-1,Федеральный округ,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,83.0


In [15]:
regions_df.to_csv('./data/regions.csv', index=False)